# H-Neurons small Qwen experiment

This notebook is configured for Colab/Kaggle with a single T4 GPU. It uses a Hugging Face model ID directly, so you do not need to download the model manually first.

In [ ]:
import os
import pathlib
import subprocess
import sys

if not pathlib.Path('H-Neurons').exists():
    subprocess.run(['git', 'clone', 'https://github.com/thunlp/H-Neurons.git'], check=True)

os.chdir('H-Neurons')
print('Working directory:', os.getcwd())

In [ ]:
# Colab/Kaggle setup. Restart the runtime if vLLM or torch dependencies require it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate', 'datasets', 'openai', 'scikit-learn', 'joblib'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True)

In [ ]:
# Use a Hugging Face model ID directly. Good for a T4 smoke test.
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'

# Set your OpenAI-compatible key only if you run extract_answer_tokens.py.
# In Colab: from google.colab import userdata; OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# In Kaggle: use Add-ons > Secrets, then read it with kaggle_secrets.
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_API_KEY')
BASE_URL = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')

OUTPUT_DIR = 'data/small_subset_qwen7b'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('MODEL_ID =', MODEL_ID)

## 1. Collect Qwen responses

This generates Qwen-specific responses and rule-based correctness labels. For training, use 5 responses per question as requested. Increase `--max_samples` if you do not get enough balanced true/false samples.

In [ ]:
collect_cmd = [
    sys.executable, 'scripts/collect_responses.py',
    '--model_path', MODEL_ID,
    '--data_path', 'data/TriviaQA/rc.nocontext/train-00000-of-00001.parquet',
    '--output_path', f'{OUTPUT_DIR}/qwen_train_samples_5resp.jsonl',
    '--sample_num', '5',
    '--max_samples', '200',
    '--judge_type', 'rule',
    '--gpu_util', '0.80',
    '--tp_size', '1',
]
subprocess.run(collect_cmd, check=True)

## 2. Extract Qwen-tokenized answer tokens

This step uses an LLM API to select answer tokens, but tokenization is done with the Qwen tokenizer through `--tokenizer_path MODEL_ID`.

In [ ]:
extract_cmd = [
    sys.executable, 'scripts/extract_answer_tokens.py',
    '--input_path', f'{OUTPUT_DIR}/qwen_train_samples_5resp.jsonl',
    '--output_path', f'{OUTPUT_DIR}/qwen_answer_tokens_train.jsonl',
    '--tokenizer_path', MODEL_ID,
    '--api_key', OPENAI_API_KEY,
    '--base_url', BASE_URL,
    '--llm_model', 'gpt-4o',
]
subprocess.run(extract_cmd, check=True)

## 3. Sample balanced train IDs

This requests 100 true + 100 false IDs, giving 200 training questions if enough consistent Qwen samples are available.

In [ ]:
sample_cmd = [
    sys.executable, 'scripts/sample_balanced_ids.py',
    '--input_path', f'{OUTPUT_DIR}/qwen_answer_tokens_train.jsonl',
    '--output_path', f'{OUTPUT_DIR}/qwen_train_qids.json',
    '--num_samples', '100',
]
subprocess.run(sample_cmd, check=True)

## Next steps

After these files are ready, run activation extraction with the same `MODEL_ID`. If exact answer-token matching is unstable, use `--locations output` as a cheaper smoke-test path.